In [1]:
import chromadb
import copy
import json
import typing as t
import uuid
import pathlib

from chromadb.utils import embedding_functions
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from constants import (
    OPENAI_API_KEY,
    VECTOR_STORE_COLLECTION,
    VECTOR_STORE_HOST,
    VECTOR_STORE_PORT
)
from data_types.pipeline import Pipeline

ModuleNotFoundError: No module named 'constants'

In [ ]:
def node_to_encoding(node):
    annotation = node["annotation"]
    node_encoding = []
    for k, v in annotation.items():
        if isinstance(v, dict):
            for key in v:
                node_encoding.append(f"{k}_{key} = {v[key]}")
        else:
            node_encoding.append(f"{k} = {v}")
    return ', '.join(node_encoding)


def pipeline_to_splits(pipeline: Pipeline) -> t.List[Pipeline]:
    splits = []
    pipeline_encoding = []
    for node in reversed(pipeline):
        node_encoding = node_to_encoding(node)
        pipeline_encoding.append(node_encoding)
        splits.append(copy.deepcopy(pipeline_encoding))
    return splits


def pipeline_to_embedding(pipeline: Pipeline):
    pipeline_splits = pipeline_to_splits(pipeline)
    pipeline_payload = (
        [str(uuid.uuid4()) for _ in range(len(pipeline_splits))],
        [json.dumps(copy.deepcopy(pipeline)) for _ in range(len(pipeline_splits))],
        [';'.join(pipeline_split) for pipeline_split in pipeline_splits]
    )
    return pipeline_payload

In [ ]:
pretrained_embeddings = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-ada-002"
)

vector_store = chromadb.HttpClient(
    host=VECTOR_STORE_HOST, 
    port=VECTOR_STORE_PORT
)

In [ ]:
vector_collection = vector_store.get_collection(VECTOR_STORE_COLLECTION)

In [ ]:
vector_collection.get()

In [ ]:
root_dir = pathlib.Path.cwd()

In [ ]:
annotated_dir, raw_dir = (
    root_dir / "data" / "eda4sum" / "annotated",
    root_dir / "data" / "eda4sum" / "raw"
)

In [ ]:
annotated_pipelines = []
for annotated_file in annotated_dir.glob('*.json'):
    with annotated_file.open('r') as f:
        annotated_pipelines.append(json.load(f))

In [ ]:
for annotated_pipeline in annotated_pipelines:
    (
        pipeline_ids,
        pipeline_documents,
        pipeline_sentences
    ) = pipeline_to_embedding(annotated_pipeline)
    vector_collection.add(
        ids=pipeline_ids,
        documents=pipeline_documents,
        embeddings=pretrained_embeddings(pipeline_sentences),
    )

In [ ]:
pipeline_collection = vector_collection.get()

In [ ]:
pipeline_collection.keys()

In [ ]:
terminal_pipeline_id, *rest_pipeline_ids = pipeline_collection['ids']
terminal_pipeline_dict, *rest_pipeline_dicts = [json.loads(pipeline_doc) for pipeline_doc in pipeline_collection['documents']]

In [ ]:
(
    terminal_pipeline_ids,
    terminal_pipeline_documents,
    terminal_pipeline_sentences
) = pipeline_to_embedding(annotated_pipeline)

terminal_annotation_embedding = pretrained_embeddings(terminal_pipeline_sentences)

terminal_response = vector_collection.query(
    query_embeddings=terminal_annotation_embedding,
    n_results=len(rest_pipeline_ids),
    include=["distances", "documents"]
)

In [ ]:
def select_attributes(pipeline):
    return f"{[node['operator'] for node in pipeline]}"

In [ ]:
print(f"Terminal pipeline: {select_attributes(terminal_pipeline_dict)}\n")

for terminal_distances, terminal_documents in zip(terminal_response['distances'], terminal_response['documents']):
    min_doc, max_doc = (
        json.loads(terminal_documents[0]),
        json.loads(terminal_documents[-1])
    )
    min_similarity, max_similarity = (
        terminal_distances[0] * 100,
        terminal_distances[-1] * 100
    )
    print(f"Min pipeline: {min_similarity}%, {select_attributes(min_doc)}")
    print(f"Max pipeline: {max_similarity}%, {select_attributes(max_doc)}\n")